# Semantic search with FAISS (PyTorch)

##  Loading and preparing the dataset

In [ ]:
# !pip install datasets evaluate transformers[sentencepiece]
# !pip install faiss-gpu

> ### Technical Note: FAISS Installation and GPU Adaptation on Windows

> **Note on Dependencies:**
> The original notebook recommends using `faiss-gpu` for vector similarity search acceleration. However, because my notebook runs in a **Windows environment using `pip**`, standard installation paths require adjustment:
> 1. **Lack of Official PyPI Support for GPU Binaries:** Meta (the maintainers of FAISS) does not officially publish pre-built GPU binaries (`faiss-gpu`) to PyPI due to package size limitations. Official GPU packages are restricted to the Conda ecosystem (via Anaconda/Miniconda) and are primarily optimized for Linux distributions.
> 2. **Environment Constraint:** Since this setup relies on a standard Python virtual environment managed via `pip` on Windows rather than Conda, installing `faiss-gpu` directly via pip fails.
> 3. **The Solution:** We fallback to **`faiss-cpu`** via `pip install faiss-cpu`.
> 
> 
> *Impact:* While the vector indexing and similarity search will now execute on the CPU, your primary deep learning model (e.g., PyTorch transformer models) can—and should—remain on the GPU for embedding generation. For most standard dataset sizes, CPU-based FAISS remains exceptionally fast and eliminates complex CUDA/Conda configuration overhead on Windows.


In [ ]:
# This code cell is there in the Chapter's notebook. 
# Instead of using this, I have used the dataset I created in previous chapter.

# from datasets import load_dataset

# issues_dataset = load_dataset("lewtun/github-issues", split="train")
# issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 2855
})

Loading the dataset that I created in previous chapter (see [notebook](../5.5%20%20Creating%20your%20own%20dataset/Creating_your_own_dataset.ipynb)):

In [1]:
from datasets import load_dataset

issues_dataset = load_dataset(
    "json",
    data_files=r"../5.5  Creating your own dataset/datasets-issues_with_comments_cleaned.json",
    split="train",
)
issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment', 'is_pull_request'],
    num_rows: 8281
})

Here we've specified the default `train` split in `load_dataset()`, so it returns a `Dataset` instead of a `DatasetDict`. The first order of business is to filter out the pull requests, as these tend to be rarely used for answering user queries and will introduce noise in our search engine. As should be familiar by now, we can use the `Dataset.filter()` function to exclude these rows in our dataset. While we're at it, let's also filter out rows with no comments, since these provide no answers to user queries:

In [3]:
issues_dataset = issues_dataset.filter(lambda x: (x["is_pull_request"] == False and len(x["comments"]) > 0))
issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment', 'is_pull_request'],
    num_rows: 2808
})

In [5]:
issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment', 'is_pull_request'],
    num_rows: 2808
})

We can see that there are a lot of columns in our dataset, most of which we don't need to build our search engine. From a search perspective, the most informative columns are `title`, `body`, and `comments`, while `html_url` provides us with a link back to the source issue. Let's use the `Dataset.remove_columns()` function to drop the rest:

In [6]:
columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns_to_keep).symmetric_difference(columns)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 2808
})

To create our embeddings we'll augment each comment with the issue's title and body, since these fields often include useful contextual information. Because our `comments` column is currently a list of comments for each issue, we need to "explode" the column so that each row consists of an `(html_url, title, body, comment)` tuple. In Pandas we can do this with the [`DataFrame.explode()` function](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.explode.html), which creates a new row for each element in a list-like column, while replicating all the other column values. To see this in action, let's first switch to the Pandas  `DataFrame` format:

In [7]:
issues_dataset.set_format("pandas")
df = issues_dataset[:]

If we inspect the first row in this `DataFrame` we can see there are four comments associated with this issue:

In [8]:
df["comments"][0].tolist()

["Reference: \n\n1. https://github.com/boostorg/beast/issues/1445\n2. `man 2 close`:\n```\n Dealing with error returns from close()\n     A careful programmer will check the return value of close(), since it is quite possible that errors on a previous write(2) operation are reported only on the final close() that releases the open file descrip‐\n     tion.  Failing to check the return value when closing a file may lead to silent loss of data.  This can especially be observed with NFS and with disk quota.\n\n     Note, however, that a failure return should be used only for diagnostic purposes (i.e., a warning to the application that there may still be I/O pending or there may have been failed I/O) or remedial purposes\n     (e.g., writing the file once more or creating a backup).\n\n     Retrying  the  close() after a failure return is the wrong thing to do, since this may cause a reused file descriptor from another thread to be closed.  This can occur because the Linux kernel always re

When we explode `df`, we expect to get one row for each of these comments. Let's check if that's the case:

In [9]:
comments_df = df.explode("comments", ignore_index=True)
comments_df.head(4)

,html_url,title,comments,body
0,https://github.com/huggingface/datasets/issues...,`Dataset.map(num_proc=N)` worker crashes with ...,Reference: \n\n1. https://github.com/boostorg/...,### Describe the bug\n\nWe hit this while runn...
1,https://github.com/huggingface/datasets/issues...,ClassLabel.str2int and int2str do not validate...,"Thanks for weighing in, but one correction so ...",### Describe the bug\n\n`ClassLabel.str2int` a...
2,https://github.com/huggingface/datasets/issues...,batch(0) silently returns one batch with the w...,Hi @shashvat-singham i did also try and repro...,### Describe the bug\n\n`Dataset.batch()` and ...
3,https://github.com/huggingface/datasets/issues...,Replace httpx with httpx2,Hi @lhoestq can I work on this,### Feature request\n\nReplace the httpx libra...


Great, we can see the rows have been replicated, with the `comments` column containing the individual comments! Now that we're finished with Pandas, we can quickly switch back to a `Dataset` by loading the `DataFrame` in memory:

In [10]:
from datasets import Dataset

comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 10812
})

Now that we have one comment per row, let's create a new `comments_length` column that contains the number of words per comment:

In [11]:
comments_dataset = comments_dataset.map(lambda x: {"comment_length": len(x["comments"].split())})

Map:   0%|          | 0/10812 [00:00<?, ? examples/s]


We can use this new column to filter out short comments, which typically include things like "cc @{user_name}" or "Thanks!" that are not relevant for our search engine. There's no precise number to select for the filter, but around 15 words seems like a good start:

In [12]:
comments_dataset = comments_dataset.filter(lambda x: x["comment_length"] > 15)
comments_dataset

Filter:   0%|          | 0/10812 [00:00<?, ? examples/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length'],
    num_rows: 7863
})

Having cleaned up our dataset a bit, let's concatenate the issue title, description, and comments together in a new `text` column. As usual, we'll write a simple function that we can pass to `Dataset.map()`:

In [13]:
def concatenate_text(examples):
    # This automatically filters out None values and joins them safely
    parts = [str(val) for val in [examples["title"], examples["body"], examples["comments"]] if val is not None]
    return {"text": " \n ".join(parts)}


comments_dataset = comments_dataset.map(concatenate_text)

Map:   0%|          | 0/7863 [00:00<?, ? examples/s]

We’re finally ready to create some embeddings! Let’s take a look.

##  Creating text embeddings

We saw in Chapter 2 that we can obtain token embeddings by using the `AutoModel` class. All we need to do is pick a suitable checkpoint to load the model from. Fortunately, there's a library called `sentence-transformers` that is dedicated to creating embeddings. As described in the library's [documentation](https://www.sbert.net/examples/applications/semantic-search/README.html#symmetric-vs-asymmetric-semantic-search), our use case is an example of _asymmetric semantic search_ because we have a short query whose answer we'd like to find in a longer document, like a an issue comment. The handy [model overview table](https://www.sbert.net/docs/pretrained_models.html#model-overview) in the documentation indicates that the `multi-qa-mpnet-base-dot-v1` checkpoint has the best performance for semantic search, so we'll use that for our application. We'll also load the tokenizer using the same checkpoint:


In [14]:
from transformers import AutoTokenizer, AutoModel

model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)


To speed up the embedding process, it helps to place the model and inputs on a GPU device, so let's do that now:

In [ ]:
import torch

device = torch.device("cuda")
model.to(device)

As we mentioned earlier, we'd like to represent each entry in our GitHub issues corpus as a single vector, so we need to "pool" or average our token embeddings in some way. One popular approach is to perform *CLS pooling* on our model's outputs, where we simply collect the last hidden state for the special `[CLS]` token. The following function does the trick for us:

In [16]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

Next, we'll create a helper function that will tokenize a list of documents, place the tensors on the GPU, feed them to the model, and finally apply CLS pooling to the outputs:

In [17]:
def get_embeddings(text_list):
    encoded_input = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

We can test the function works by feeding it the first text entry in our corpus and inspecting the output shape:

In [18]:
embedding = get_embeddings(comments_dataset["text"][0])
embedding.shape

torch.Size([1, 768])

Great, we've converted the first entry in our corpus into a 768-dimensional vector! We can use `Dataset.map()` to apply our `get_embeddings()` function to each row in our corpus, so let's create a new `embeddings` column as follows:

In [19]:
embeddings_dataset = comments_dataset.map(lambda x: {"embeddings": get_embeddings(x["text"]).detach().cpu().numpy()[0]})

Map:   0%|          | 0/7863 [00:00<?, ? examples/s]

Notice that we've converted the embeddings to NumPy arrays -- that's because 🤗 Datasets requires this format when we try to index them with FAISS, which we'll do next.



## Using FAISS for efficient similarity search


Now that we have a dataset of embeddings, we need some way to search over them. To do this, we'll use a special data structure in 🤗 Datasets called a _FAISS index_. [FAISS](https://faiss.ai/) (short for Facebook AI Similarity Search) is a library that provides efficient algorithms to quickly search and cluster embedding vectors.

The basic idea behind FAISS is to create a special data structure called an _index_ that allows one to find which embeddings are similar to an input embedding. Creating a FAISS index in 🤗 Datasets is simple -- we use the `Dataset.add_faiss_index()` function and specify which column of our dataset we'd like to index:

In [22]:
import faiss

print(faiss.__version__)

1.15.0


In [23]:
embeddings_dataset.add_faiss_index(column="embeddings")

  0%|          | 0/8 [00:00<?, ?it/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length', 'text', 'embeddings'],
    num_rows: 7863
})

We can now perform queries on this index by doing a nearest neighbor lookup with the `Dataset.get_nearest_examples()` function. Let's test this out by first embedding a question as follows:

In [24]:
question = "How can I load a dataset offline?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

(1, 768)

Just like with the documents, we now have a 768-dimensional vector representing the query, which we can compare against the whole corpus to find the most similar embeddings:

In [25]:
scores, samples = embeddings_dataset.get_nearest_examples("embeddings", question_embedding, k=5)

The `Dataset.get_nearest_examples()` function returns a tuple of scores that rank the overlap between the query and the document, and a corresponding set of samples (here, the 5 best matches). Let's collect these in a `pandas.DataFrame` so we can easily sort them:

In [26]:
import pandas as pd

samples_df = pd.DataFrame.from_dict(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)

Now we can iterate over the first few rows to see how well our query matched the available comments:

In [27]:
for _, row in samples_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.scores}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()

COMMENT: Requiring online connection is a deal breaker in some cases unfortunately so it'd be great if offline mode is added similar to how `transformers` loads models offline fine.

@mandubian's second bullet point suggests that there's a workaround allowing you to use your offline (custom?) dataset with `datasets`. Could you please elaborate on how that should look like?
SCORE: 25.505016326904297
TITLE: Discussion using datasets in offline mode
URL: https://github.com/huggingface/datasets/issues/824

COMMENT: The local dataset builders (csv, text , json and pandas) are now part of the `datasets` package since #1726 :)
You can now use them offline
```python
datasets = load_dataset('text', data_files=data_files)
```

We'll do a new release soon
SCORE: 24.55554962158203
TITLE: Discussion using datasets in offline mode
URL: https://github.com/huggingface/datasets/issues/824

COMMENT: I opened a PR that allows to reload modules that have already been loaded once even if there's no interne

 Our second hit seems to match the query. Not bad!😏

> ✏️ **Try it out!** Create your own query and see whether you can find an answer in the retrieved documents. You might have to increase the `k` parameter in `Dataset.get_nearest_examples()` to broaden the search.

In [28]:
### Trying my own query

question = "How to upload the datasets to Hub?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

scores, samples = embeddings_dataset.get_nearest_examples("embeddings", question_embedding, k=5)
samples_df = pd.DataFrame.from_dict(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)

for _, row in samples_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.scores}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()

COMMENT: Hi @dnaveenr ! We can contact the authors to see if they are interested in hosting the dataset on the Hub. In the meantime, feel free to work on a script with manual download.
SCORE: 30.916057586669922
TITLE: Add IEMOCAP dataset
URL: https://github.com/huggingface/datasets/issues/3285

COMMENT: had a discussion with @neurolabusc and here's a quick wrap-up:
 - BIDS support would be huge (@bruAristimunha would be great if we could catch up on that)
 - DICOM support as well, but that might be harder due to a lot of variety in how headers are handled, vendor specifics etc. So to have a reliable pipeline to interact with whole folders of DICOM files (including metadata) would require a lot of work and a lot of testing. Therefore I set https://github.com/huggingface/datasets/pull/7835 back to draft mode. But there are tools that ease the way, especially https://github.com/ImagingDataCommons/highdicom (or potentially https://github.com/QIICR/dcmqi). 
 - Getting users would help in or

The 4th comment looks to be the answer:

```text
COMMENT: > > You can use `.push_to_hub("<username>/<repo>")` to push a `Dataset` to the Hub.
```

Again, not bad!😏😏


## Let's write an interactive function!💻🧠

I have written a function for the query matching and generating top-k answers.

### Overview

The `search_dataset_faiss` function is a self-contained utility designed to query a Hugging Face dataset indexed with FAISS. It prompts the user interactively for a search question and the desired number of results, generates the query embedding, searches the vector database, and prints formatted, sorted results.

### Function Signature

```python
def search_dataset_faiss(embeddings_dataset, get_embeddings_func)

```

### Parameters

| Parameter | Type | Description |
| --- | --- | --- |
| `embeddings_dataset` | `Dataset` / `DatasetDict` | A Hugging Face dataset instance that has been indexed using `add_faiss_index()`. |
| `get_embeddings_func` | `callable` | A function that takes a list of strings (queries) and returns a PyTorch tensor of their corresponding embeddings. |


### Interactive Prompts

1. **Search Query (`question`)**: Prompts the user to enter a natural language question or search term.
2. **Number of Results (`k`)**: Prompts the user naturally (*"How many relevant results would you like to see?"*) with built-in validation to ensure a positive integer is provided (defaulting safely if needed).

In [29]:
import pandas as pd
import torch


def search_dataset_faiss(embeddings_dataset, get_embeddings_func):
    """Asks the user for a search query and the number of results to return,

    then queries the Faiss index and prints the matching samples.
    """
    # 1. Get user inputs with natural prompts
    question = input("Enter your search query or question: ").strip()

    while True:
        try:
            k_input = input("How many relevant results would you like to see? (e.g., 5): ").strip()
            k = int(k_input) if k_input else 5  # Default to 5 if blank
            if k > 0:
                break
            print("Please enter a number greater than 0.")
        except ValueError:
            print("Invalid input. Please enter a valid whole number.")

    print(f"\nSearching for: '{question}' (Top {k} results)...\n" + "=" * 50)

    # 2. Generate and format question embedding
    question_embedding = get_embeddings_func([question]).cpu().detach().numpy()

    # 3. Query the Faiss index
    scores, samples = embeddings_dataset.get_nearest_examples("embeddings", question_embedding, k=k)

    # 4. Process and sort results using pandas
    samples_df = pd.DataFrame.from_dict(samples)
    samples_df["scores"] = scores
    samples_df.sort_values("scores", ascending=False, inplace=True)

    # 5. Print results nicely
    for _, row in samples_df.iterrows():
        print(f"COMMENT: {row.comments}")
        print(f"SCORE: {row.scores}")
        print(f"TITLE: {row.title}")
        print(f"URL: {row.html_url}")
        print("=" * 50)
        print()

### Example Usage

```python
# Assuming embeddings_dataset and get_embeddings are already defined in your session:
search_dataset_faiss(embeddings_dataset, get_embeddings)

```

In [ ]:
search_dataset_faiss(embeddings_dataset, get_embeddings)

# I ran with the query: How to add a column in the dataset?
# k = 5


Searching for: 'How to add a column in the dataset?' (Top 5 results)...
COMMENT: For Spark it looks to be pretty straightforward as well https://spark.apache.org/docs/latest/sql-pyspark-pandas-with-arrow.html but looks to be having a dependency to Spark is necessary, then nevermind we can skip it
SCORE: 30.928638458251953
TITLE: [Feature] More dataset outputs
URL: https://github.com/huggingface/datasets/issues/3

COMMENT: You can use the `remove_columns` parameter in `map` to avoid duplicating the columns (and save disk space) and then concatenate the original dataset with the map result:
```python
from datasets import concatenate_datasets
# dummy example
ds_new = ds.map(lambda x: {"new_col": x["col"] + 2}, remove_columns=ds.column_names)
ds_combined = concatenate_datasets([ds, ds_new], axis=1)
```

Doing this automatically is hard to implement efficiently unless we know ahead of time which existing columns will be modified by a `map` transform. We have this info when `input_columns` 

....and it's working!🥳

Trying again with another query:

In [31]:
search_dataset_faiss(embeddings_dataset, get_embeddings)


Searching for: 'How to rename a column?' (Top 10 results)...
COMMENT: The task templates API has been deprecated (will be removed in version 3.0), so I'm closing this issue.
SCORE: 40.835880279541016
TITLE: label_column='labels' in datasets.TextClassification and 'label' or 'label_ids' in transformers.DataColator
URL: https://github.com/huggingface/datasets/issues/5419

COMMENT: Hi! Thanks for pointing out this inconsistency. Changing the default value at this point is probably not worth it, considering we've started discussing the state of the task API internally - we will most likely deprecate the current one and replace it with a more robust solution that relies on the `train_eval_index` field stored in the YAML section of the dataset cards.
SCORE: 40.835880279541016
TITLE: label_column='labels' in datasets.TextClassification and 'label' or 'label_ids' in transformers.DataColator
URL: https://github.com/huggingface/datasets/issues/5419

COMMENT: @alvarobartt Thanks. My use case was